# Correlation Functions and Coarsening Dynamics

Spatial correlations and domain growth are two sides of the same coin. The equilibrium correlation function $G(r) = \langle \vec{s}_0 \cdot \vec{s}_r \rangle$ encodes the spatial structure of the ordered or disordered phase at a given temperature. When a system is suddenly quenched from a disordered initial condition into the ordered regime, domains of the equilibrium phase nucleate and grow through coarsening. The characteristic length scale of these domains, $R(t)$, increases with Monte Carlo time according to power laws whose exponents depend on the symmetry of the order parameter and the nature of the topological defects involved [[1]](#Bibliography) [[2]](#Bibliography).

This notebook presents both topics side by side for the three VibeSpin models (Ising, XY, and $q$-state clock), connecting the equilibrium correlation structure in Part I to the non-equilibrium coarsening dynamics in Part II. Part I loads or computes equilibrium $G(r)$ at representative temperatures. Part II loads or computes ordering kinetics traces showing how the domain size $R(t)$ evolves after a quench, with power-law fits that extract the growth exponent.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from models.ising_model import IsingSimulation
from models.xy_model import XYSimulation
from models.clock_model import ClockSimulation
from utils.equilibration import convergence_equilibrate
from utils.observables import get_averaged_correlation


def _find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'results').exists():
            return candidate
    return cwd


REPO_ROOT = _find_repo_root()
TC_ISING = 2.0 / np.log(1.0 + np.sqrt(2.0))
T_BKT = 0.893

PALETTE = {
    'ising': '#4878CF',
    'xy': '#6ACC65',
    'clock': '#EE854A',
    'ferro': '#4878CF',
    'crit': '#D65F5F',
    'para': '#6ACC65',
}

plt.rcParams['figure.dpi'] = 120

print(f'T_c (Ising, exact) = {TC_ISING:.4f}')
print(f'T_BKT (XY, theoretical) = {T_BKT}')
print(f'Repository root: {REPO_ROOT}')

---

## Part I: Equilibrium Correlation Functions

The equilibrium spin-spin correlation function is computed via the structure-factor route: the Fourier transform of the spin field yields $S(\mathbf{k})$, and radial averaging of the inverse transform gives $G(r)$ normalized to $G(0) = 1$. This Wiener-Khinchin approach avoids $O(N^2)$ real-space pair counting.

### Ising Correlations

The Ising model has three qualitatively different correlation regimes. In the ferromagnetic phase ($T < T_c$) correlations saturate to a long-range plateau. At the critical point ($T = T_c$) the decay is algebraic: $G(r) \sim r^{-\eta}$ with the exact exponent $\eta = 1/4$ [[3]](#Bibliography). In the paramagnetic phase ($T > T_c$) the decay is exponential with a correlation length $\xi(T)$ that diverges as $T \to T_c^+$.

In [ ]:
ising_corr_cache = REPO_ROOT / 'results' / 'ising' / 'correlation_comparison.npz'
ising_corr_loaded = False


def _compute_ising_correlations(
    *,
    size: int = 40,
    total_steps: int = 1_500,
    sample_interval: int = 15,
    eq_probe: int = 150,
    eq_max: int = 6_000,
) -> dict:
    temps = {
        'ferro': 0.5 * TC_ISING,
        'crit': TC_ISING,
        'para': 1.5 * TC_ISING,
    }
    result = {}
    for label, t in temps.items():
        sim_r = IsingSimulation(size=size, temp=float(t), update='checkerboard', init_state='random', seed=500)
        sim_o = IsingSimulation(size=size, temp=float(t), update='checkerboard', init_state='ordered', seed=500)
        convergence_equilibrate(sim_r, sim_o, chunk_size=eq_probe, max_steps=eq_max)
        r, G = get_averaged_correlation(sim=sim_r, total_steps=total_steps, sample_interval=sample_interval)
        result[label] = (np.asarray(r, dtype=float), np.asarray(G, dtype=float), float(t))
    return result


if ising_corr_cache.exists():
    d = np.load(ising_corr_cache)
    if all(k in d.files for k in ('r', 'G_ferro', 'G_crit', 'G_para', 'T_ferro', 'T_crit', 'T_para')):
        r_ising = np.asarray(d['r'], dtype=float)
        ising_corr = {
            'ferro': (r_ising, np.asarray(d['G_ferro'], dtype=float), float(d['T_ferro'])),
            'crit': (r_ising, np.asarray(d['G_crit'], dtype=float), float(d['T_crit'])),
            'para': (r_ising, np.asarray(d['G_para'], dtype=float), float(d['T_para'])),
        }
        ising_corr_source = f'Loaded Ising correlations from cache'
        ising_corr_loaded = True

if not ising_corr_loaded:
    ising_corr = _compute_ising_correlations()
    ising_corr_source = 'Computed fallback Ising correlations (L=40)'

print(ising_corr_source)

The log-log plot below shows $G(r)$ at the three representative Ising temperatures. The critical curve should appear approximately straight with slope $-1/4$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))

for label, (r, G, t) in ising_corr.items():
    mask = (r > 0) & (G > 0)
    regime = {'ferro': 'Ferro', 'crit': 'Critical', 'para': 'Para'}[label]
    ax.loglog(r[mask], G[mask], 'o-', ms=3, lw=1.5, color=PALETTE[label],
              label=rf'{regime} ($T = {t:.3f}$)')

r_guide = np.linspace(1, 15, 50)
ax.loglog(r_guide, 0.8 * r_guide ** (-0.25), '--', color='0.4', lw=1.0, label=r'$r^{-1/4}$ guide')

ax.set_xlabel('Distance $r$')
ax.set_ylabel('$G(r)$')
ax.set_title('Ising model: equilibrium correlation function')
ax.grid(alpha=0.25, which='both')
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

### XY Correlations

The XY model exhibits algebraic correlations $G(r) \sim r^{-\eta(T)}$ throughout the quasi-ordered phase $T < T_{\mathrm{BKT}}$, with the exponent $\eta$ increasing continuously from near zero at low temperature to $\eta(T_{\mathrm{BKT}}) = 1/4$ at the BKT transition. Above $T_{\mathrm{BKT}}$ the correlation function crosses over to exponential decay [[4]](#Bibliography).

In [ ]:
xy_corr_cache = REPO_ROOT / 'results' / 'xy' / 'correlation_comparison.npz'
xy_corr_loaded = False


def _compute_xy_correlations(
    *,
    size: int = 40,
    total_steps: int = 1_500,
    sample_interval: int = 15,
    eq_probe: int = 150,
    eq_max: int = 6_000,
) -> dict:
    temps = {'low': 0.5 * T_BKT, 'high': 1.5 * T_BKT}
    result = {}
    for label, t in temps.items():
        sim_r = XYSimulation(size=size, temp=float(t), update='checkerboard', init_state='random', seed=510)
        sim_o = XYSimulation(size=size, temp=float(t), update='checkerboard', init_state='ordered', seed=510)
        convergence_equilibrate(sim_r, sim_o, chunk_size=eq_probe, max_steps=eq_max)
        r, G = get_averaged_correlation(sim=sim_r, total_steps=total_steps, sample_interval=sample_interval)
        result[label] = (np.asarray(r, dtype=float), np.asarray(G, dtype=float), float(t))
    return result


if xy_corr_cache.exists():
    d = np.load(xy_corr_cache)
    if all(k in d.files for k in ('r_low', 'G_low', 'r_high', 'G_high')):
        T_LOW_XY = float(d['T_low']) if 'T_low' in d.files else 0.5 * T_BKT
        T_HIGH_XY = float(d['T_high']) if 'T_high' in d.files else 1.5 * T_BKT
        xy_corr = {
            'low': (np.asarray(d['r_low'], dtype=float), np.asarray(d['G_low'], dtype=float), T_LOW_XY),
            'high': (np.asarray(d['r_high'], dtype=float), np.asarray(d['G_high'], dtype=float), T_HIGH_XY),
        }
        xy_corr_source = 'Loaded XY correlations from cache'
        xy_corr_loaded = True

if not xy_corr_loaded:
    xy_corr = _compute_xy_correlations()
    xy_corr_source = 'Computed fallback XY correlations (L=40)'

print(xy_corr_source)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for label, (r, G, t) in xy_corr.items():
    mask = (r > 0) & (G > 0)
    color = PALETTE['ferro'] if label == 'low' else PALETTE['crit']
    regime = 'Below' if label == 'low' else 'Above'
    axes[0].loglog(r[mask], G[mask], 'o-', ms=3, lw=1.5, color=color,
                   label=rf'{regime} $T_{{\mathrm{{BKT}}}}$ ($T = {t:.3f}$)')
    axes[1].semilogy(r[mask], G[mask], 'o-', ms=3, lw=1.5, color=color,
                     label=rf'{regime} $T_{{\mathrm{{BKT}}}}$ ($T = {t:.3f}$)')

axes[0].set_xlabel('Distance $r$')
axes[0].set_ylabel('$G(r)$')
axes[0].set_title('Log-log (power-law is linear)')
axes[0].grid(alpha=0.25, which='both')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('Distance $r$')
axes[1].set_ylabel('$G(r)$')
axes[1].set_title('Semi-log (exponential is linear)')
axes[1].grid(alpha=0.25, which='both')
axes[1].legend(fontsize=8)

fig.suptitle('XY model: equilibrium correlation function', y=1.02)
fig.tight_layout()
plt.show()

---

## Part II: Coarsening Dynamics

When a system is quenched instantaneously from infinite temperature ($T = \infty$) to a temperature deep inside the ordered phase, the initial random configuration breaks up into domains of locally ordered regions separated by walls (Ising) or by topological defects such as vortices (XY) and domain boundaries (clock). The characteristic domain size $R(t)$ grows according to power laws that depend on whether the order parameter is discrete or continuous and on the nature of the dominant defects.

For the Ising model with non-conserved dynamics, the Allen-Cahn growth law gives $R(t) \sim t^{1/2}$ [[1]](#Bibliography). For the XY model, vortex annihilation drives coarsening with the same $t^{1/2}$ law at sufficiently late times, though early-time crossover behavior is common [[2]](#Bibliography). The clock model interpolates: for small $q$ the discrete domain-wall dynamics resemble Ising coarsening, while for large $q$ the continuous-like vortex dynamics resemble the XY case.

The VibeSpin kinetics scripts measure three length-scale proxies: $R_{S(k)}$ from the structure factor peak, $R_\xi$ from the correlation function second moment, and a model-specific third metric (excess energy for Ising, vortex density for XY/clock). Power-law fits $R(t) \sim t^n$ are performed in the late-time regime.

### Load Kinetics Data

The precomputed kinetics data comes from `scripts/{model}/ordering_kinetics.py`. The fallback computes a short kinetics run using `utils.kinetics_helpers.run_ordering_kinetics`. Since kinetics simulations require random-site updates (physical dynamics), the fallback necessarily uses smaller lattices and fewer time steps.

In [ ]:
kinetics_data = {}

for model_name, cache_path in [
    ('Ising', REPO_ROOT / 'results' / 'ising' / 'ordering_kinetics.npz'),
    ('XY', REPO_ROOT / 'results' / 'xy' / 'ordering_kinetics.npz'),
    ('Clock', REPO_ROOT / 'results' / 'clock' / 'ordering_kinetics.npz'),
]:
    if cache_path.exists():
        d = np.load(cache_path, allow_pickle=True)
        if all(k in d.files for k in ('t', 'R_sk', 'R_xi', 'exponent_R_sk', 'exponent_xi')):
            kinetics_data[model_name] = {
                't': np.asarray(d['t'], dtype=float),
                'R_sk': np.asarray(d['R_sk'], dtype=float),
                'R_xi': np.asarray(d['R_xi'], dtype=float),
                'third_metric': np.asarray(d['third_metric'], dtype=float) if 'third_metric' in d.files else None,
                'third_label': str(d['third_metric_label']) if 'third_metric_label' in d.files else None,
                'exp_sk': float(d['exponent_R_sk']),
                'exp_xi': float(d['exponent_xi']),
                'exp_third': float(d['exponent_third']) if 'exponent_third' in d.files else None,
                'pre_sk': float(d['prefactor_R_sk']),
                'pre_xi': float(d['prefactor_xi']),
                'fit_min': int(d['fit_min']),
                'size': int(d['size']),
                'temp': float(d['temp']),
            }
            print(f'{model_name}: loaded from cache (L={int(d["size"])}, T={float(d["temp"]):.2f})')
        else:
            print(f'{model_name}: cache incomplete, skipping')
    else:
        print(f'{model_name}: cache not found, skipping')

if not kinetics_data:
    print('\nNo kinetics data found. Run the ordering_kinetics scripts to populate the cache.')

### Domain Growth Curves

The domain-size proxies $R_{S(k)}$ and $R_\xi$ are plotted on log-log axes for each model that has cached data. A straight line on these axes indicates power-law growth $R \sim t^n$. The fitted exponents from the late-time regime are shown in the legend. The Allen-Cahn prediction $n = 1/2$ is the expected asymptotic result for both Ising (curvature-driven domain walls) and XY (vortex annihilation) [[1]](#Bibliography) [[2]](#Bibliography).

In [ ]:
if kinetics_data:
    n_models = len(kinetics_data)
    fig, axes = plt.subplots(1, n_models, figsize=(5.5 * n_models, 5), squeeze=False)
    model_colors = {'Ising': PALETTE['ising'], 'XY': PALETTE['xy'], 'Clock': PALETTE['clock']}

    for idx, (model_name, kd) in enumerate(kinetics_data.items()):
        ax = axes[0, idx]
        t = kd['t']
        mask = t > 0

        ax.loglog(t[mask], kd['R_sk'][mask], '-', color=model_colors[model_name], lw=2,
                  label=rf'$R_{{S(k)}}$ ($n = {kd["exp_sk"]:.3f}$)')
        ax.loglog(t[mask], kd['R_xi'][mask], '--', color=model_colors[model_name], lw=2, alpha=0.7,
                  label=rf'$R_\xi$ ($n = {kd["exp_xi"]:.3f}$)')

        fit_min = kd['fit_min']
        t_fit = np.linspace(fit_min, t.max(), 50)
        ax.loglog(t_fit, kd['pre_sk'] * t_fit ** kd['exp_sk'], ':',
                  color='0.3', lw=1.0, label='Power-law fit')

        ax.axvline(fit_min, color='0.5', ls=':', lw=0.8)
        ax.set_xlabel('Monte Carlo time $t$')
        ax.set_ylabel('Domain size $R(t)$')
        ax.set_title(f'{model_name} (L={kd["size"]}, T={kd["temp"]:.2f})')
        ax.grid(alpha=0.25, which='both')
        ax.legend(fontsize=8)

    fig.suptitle('Coarsening: domain growth after infinite-temperature quench', y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print('No kinetics data available for plotting.')

### Exponent Comparison

The table below collects the fitted growth exponents from all available models and compares them to the Allen-Cahn prediction of $n = 1/2$.

In [ ]:
if kinetics_data:
    header = f'{"Model":<10} {"R_S(k) exponent":>16} {"R_xi exponent":>15} {"L":>5} {"T":>6}'
    print(header)
    print('-' * len(header))
    for model_name, kd in kinetics_data.items():
        print(f'{model_name:<10} {kd["exp_sk"]:>16.4f} {kd["exp_xi"]:>15.4f} {kd["size"]:>5} {kd["temp"]:>6.2f}')
    print(f'\nAllen-Cahn prediction: n = 0.5')
else:
    print('No kinetics data available.')

## Synthesis

The equilibrium correlation functions in Part I characterize the spatial structure that coarsening dynamics in Part II must build. The Ising model's exponential-above / power-law-at / long-range-below $T_c$ pattern is the sharpest, reflecting the discrete $\mathbb{Z}_2$ symmetry and second-order transition. The XY model's algebraic quasi-order below $T_{\mathrm{BKT}}$ produces a qualitatively different correlation signature with a continuously varying exponent. The clock model interpolates between these limits depending on $q$.

In the coarsening regime, all three models approach the $R(t) \sim t^{1/2}$ Allen-Cahn law at late times, consistent with curvature-driven domain-wall or vortex annihilation dynamics. Deviations at early times reflect the transient regime before the dominant growth mechanism establishes itself, as well as finite-size and measurement-resolution effects. The agreement across models with different symmetry classes illustrates the universality of non-conserved coarsening dynamics in two dimensions [[1]](#Bibliography) [[2]](#Bibliography).

## Bibliography

[[1]](#Bibliography) A. J. Bray, "Theory of phase-ordering kinetics," *Advances in Physics* 51, 481 (2002). [arXiv:cond-mat/0205256](https://arxiv.org/abs/cond-mat/0205256)

[[2]](#Bibliography) A. J. Bray, A. J. Briant, and D. K. Jervis, "Breakdown of Scaling in the Nonequilibrium Critical Dynamics of the Two-Dimensional XY Model," *Physical Review Letters* 84, 1503 (2000). [APS](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.84.1503)

[[3]](#Bibliography) B. M. McCoy and T. T. Wu, *The Two-Dimensional Ising Model*, Harvard University Press (1973); see also L. Onsager, "Crystal Statistics. I. A Two-Dimensional Model with an Order-Disorder Transition," *Physical Review* 65, 117 (1944). [APS](https://journals.aps.org/pr/abstract/10.1103/PhysRev.65.117)

[[4]](#Bibliography) J. M. Kosterlitz and D. J. Thouless, "Ordering, metastability and phase transitions in two-dimensional systems," *Journal of Physics C* 6, 1181 (1973). [IoP Open Access](https://iopscience.iop.org/article/10.1088/0022-3719/6/7/010)